<a href="https://colab.research.google.com/github/Amitabh-Phule/GenAi/blob/main/Exp4_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Design and development of a Retrieval-Augmented Generation (RAG) based question answering chat system.**
#Objectives:
1. To understand RAG architecture
2. To build a retrieval-based QA chatbot

# 1. Install Dependencies

In [2]:
!pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu pypdf

# 2. Import Libraries

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from google.colab import files, userdata

# 3. Load GROQ API Key

In [4]:
GROQ_API_KEY = userdata.get('Lab')

# 4. Upload PDF Document

In [5]:
print("Upload your PDF:")
uploaded = files.upload()

for fn in uploaded.keys():
    pdf_path = fn

print(f"Loaded file: {pdf_path}")

Upload your PDF:


Saving Unit-V.pdf to Unit-V (4).pdf
Loaded file: Unit-V (4).pdf


# 5. Load PDF Content

In [6]:
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Total pages: {len(documents)}")

Total pages: 66


# 6. Text Chunking

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
texts = text_splitter.split_documents(documents)
print(f"Total chunks: {len(texts)}")

Total chunks: 69


# 7. Embeddings

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 8. Vector Store (FAISS)

In [9]:
vector_store = FAISS.from_documents(texts, embeddings)

# 9. Load GROQ LLM

In [10]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

# 10. Retrieval QA Chain

In [11]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever()
)

# 11. Chat Loop

In [12]:
print("\n✅ RAG Chatbot Ready (type 'exit' to stop)\n")

while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("Exiting chatbot...")
        break

    response = qa_chain.run(query)
    print("Bot:", response)


✅ RAG Chatbot Ready (type 'exit' to stop)

You: exit
Exiting chatbot...
